# 📖 Notebook 1 — What Is a CDN?

**A CDN (Content Delivery Network)** is a bunch of servers spread around the
world that keep copies of your content *close to your users*. Instead of every
visitor in Tokyo reaching all the way to your server in New York, they reach
a nearby **edge server** that already has a copy.

Think of it like this:

> 🍕 Your origin server is the main pizza kitchen in New York. It makes
> *every* pizza from scratch — tasty, but slow if you live in Tokyo.
> Edge servers are freezers in every city. The first Tokyo customer waits
> for a shipment from New York, but every customer after that gets a pizza
> straight from the local freezer.

## Learning objectives

By the end of this notebook you will:

1. See the three roles in a CDN: **origin**, **edge**, and **client**.
2. Measure the latency difference between hitting the origin and hitting an edge.
3. Understand what *cache hit*, *cache miss*, and *cache warming* mean —
   by seeing them happen in real HTTP response headers.


## 🛠️ Setup

**Step 1 — Start the lab infrastructure** (from the `01-foundations/cdn/` directory):

```bash
docker compose up -d --build
```

This starts three containers:

| Service | Role                | URL                    |
|---------|---------------------|------------------------|
| origin  | Slow FastAPI server | http://localhost:8000  |
| edge1   | nginx edge POP #1   | http://localhost:8081  |
| edge2   | nginx edge POP #2   | http://localhost:8082  |

**Step 2 — Install Python deps**:

```bash
uv sync
```

**Step 3 — Select the kernel**: in VS Code, click the kernel picker in the
top-right of this notebook and choose the `.venv` interpreter. If it doesn't
show up, reload the window (`Cmd+Shift+P` → *Reload Window*) and try again.


In [ ]:
import time
import statistics
import httpx

ORIGIN = "http://localhost:8000"
EDGE1  = "http://localhost:8081"
EDGE2  = "http://localhost:8082"

def timed_get(url: str, **kwargs) -> tuple[float, httpx.Response]:
    """Return (elapsed_ms, response) for a single GET."""
    t0 = time.perf_counter()
    r = httpx.get(url, timeout=10.0, **kwargs)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    return elapsed_ms, r

def show(url: str, elapsed_ms: float, r: httpx.Response) -> None:
    cache = r.headers.get("x-cache-status", "-")
    edge  = r.headers.get("x-edge-server", "-")
    print(f"{url:35s}  {elapsed_ms:7.1f} ms  cache={cache:8s}  edge={edge}")


## 👀 Meet the origin

The origin is a FastAPI app that sleeps **500 ms** before returning any asset.
That sleep stands in for everything that makes a real origin slow: distance,
disk I/O, DB queries, CPU load. Let's hit it a few times and confirm it's
consistently slow.

In [ ]:
# Warm up the TCP connection first, then time five origin requests.
httpx.get(f"{ORIGIN}/health")

times = []
for i in range(5):
    ms, r = timed_get(f"{ORIGIN}/assets/hello.txt")
    times.append(ms)
    show(f"ORIGIN  try #{i+1}", ms, r)

print(f"\nmedian = {statistics.median(times):.1f} ms  (every request pays the 500ms toll)")

### ☝️ What just happened?

Every single request took ~500 ms. There is **no memory**: request #5 is
just as slow as request #1. The server has no idea it already answered the
exact same question four times.

This is the *BAD* version: **every request hits the origin**.

## 🏎️ BETTER: add one edge cache

`edge1` is an nginx container configured as a caching reverse proxy. The
magic lines in its config are:

```nginx
proxy_cache_path /var/cache/nginx/edge ...;
proxy_cache edge;
add_header X-Cache-Status $upstream_cache_status;
```

That means: *when a request comes in, look in my on-disk cache first. If
it's there (HIT) serve it immediately. Otherwise fetch from the origin,
store it, and remember it for next time (MISS).*

In [ ]:
# Use a fresh URL so we can watch the MISS → HIT transition.
url = f"{EDGE1}/assets/hello.txt?demo=1"

for i in range(5):
    ms, r = timed_get(url)
    show(f"EDGE1   try #{i+1}", ms, r)

### ☝️ What just happened?

- **Request #1**: `X-Cache-Status: MISS`. edge1 had nothing cached, so it
  forwarded the request to the origin, waited the 500 ms, and kept a copy.
- **Requests #2–#5**: `X-Cache-Status: HIT`. Served from edge1's local disk
  in a millisecond or two. The origin never even knew about them.

The jump from ~500 ms to ~2 ms is the entire value proposition of a CDN in
one measurement.

## 🌍 BEST: multiple edges near users

One edge helps users near *that* edge. But a user far from edge1 still pays
the network cost to reach it. Real CDNs solve this by deploying edges in
**many Points of Presence (PoPs)**. The user is routed (by DNS or anycast)
to the *nearest* edge.

We can't simulate geography on one laptop, but we can run two edges and
show that they maintain **independent caches** — a MISS on edge1 does not
warm edge2.

In [ ]:
# Use a brand-new cache key so neither edge has it yet.
key = f"?run={int(time.time())}"
url1 = f"{EDGE1}/assets/hello.txt{key}"
url2 = f"{EDGE2}/assets/hello.txt{key}"

print("First request to each edge (both will MISS — cold caches):")
show("edge1 #1", *timed_get(url1))
show("edge2 #1", *timed_get(url2))

print("\nSecond request to each edge (both should HIT — warmed):")
show("edge1 #2", *timed_get(url1))
show("edge2 #2", *timed_get(url2))

Each edge keeps its **own** copy of the asset. That's a real
property of CDNs: 200 PoPs means the first visitor in each region pays the
"warming" cost, but then every subsequent visitor gets the fast path.

## 📦 Recap

| Scenario                  | Avg latency | Origin load           |
|---------------------------|-------------|-----------------------|
| BAD — origin only         | ~500 ms     | 1 request per user    |
| BETTER — single edge      | ~500 ms first time, ~2 ms after | 1 request per *asset* |
| BEST — multiple edges     | ~2 ms for users near any warm edge | 1 request per asset *per edge* |

You just recreated, at toy scale, what Cloudflare / Fastly / CloudFront
do for a living.

➡️ Next up: [Notebook 2 — Push vs Pull CDN](./02_push_vs_pull_cdn.ipynb)